In [ ]:
### 심층 신경망



## 2개의 층

import keras

(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

from sklearn.model_selection import train_test_split

train_scaled = train_input / 255.0 # 픽셀값 전처리 기법
train_scaled = train_scaled.reshape(-1, 28 * 28) # 입력 데이터로써 데이터 차원 변환

# 교차 검증 세트 분리
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size = 0.2, random_state = 42)

# 은닉층 - 활성화 함수를 활용한 비선형 연산 추가

inputs = keras.layers.Input(shape = (28 * 28,))
dense1 = keras.layers.Dense(100, activation = 'sigmoid')
dense2 = keras.layers.Dense(10, activation = 'softmax')

model = keras.Sequential([inputs, dense1, dense2])

model.summary() # 모델명, 층의 이름/클래스/출력 크기/모델 파라미터 개수 정보 호출
# 케라스 fit() 메서드 => 미니배치 경사 하강법 수행



## 층을 추가하는 다른 방법

# 방법 1. Sequential 메서드 생성자 내부에 층 직접 추가

model = keras.Sequential([keras.layers.Input(shape = (28 * 28,)),
                          keras.layers.Dense(100, activation = 'sigmoid', name = '은닉층'),
                          keras.layers.Dense(10, activation = 'softmax', name = '출력층')
                          ], name = '패션 MNIST 모델')

model.summary() # 방법 1 장점 - 추가하는 층 & 층 이름 파악하기 쉽다


# 방법 2. .add() 메서드 활용

model = keras.Sequential()
model.add(keras.layers.Input(shape = (28 * 28,)))
model.add(keras.layers.Dense(100, activation = 'sigmoid'))
model.add(keras.layers.Dense(10, activation = 'softmax'))

model.summary()
model.compile(loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_scaled, train_target, epochs = 5)


## 렐루(ReLU) 함수

model = keras.Sequential()
model.add(keras.layers.Input(shape = (28, 28)))
model.add(keras.layers.Flatten()) # 입력 데이터를 1차원으로 펼치기 역할
model.add(keras.layers.Dense(100, activation = 'relu'))
model.add(keras.layers.Dense(10, activation = 'softmax'))

model.summary()


(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
train_scaled = train_input / 255.0 # 픽셀값 전처리 기법
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size = 0.2, random_state = 42)

model.compile(loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_scaled, train_target, epochs = 5)



## 고급 옵티마이저 - Adam 객체 훈련

model = keras.Sequential()
model.add(keras.layers.Input(shape = (28, 28)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(100, activation = 'relu'))
model.add(keras.layers.Dense(10, activation = 'softmax'))

model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_scaled, train_target, epochs = 5)
model.evaluate(val_scaled, val_target)

In [ ]:
### 은닉층 개수에 따른 딥러닝 모델 성능 변화 관찰 예제

import keras

## 훈련-검증-테스트 세트 추출
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
from sklearn.model_selection import train_test_split
train_scaled = train_input / 255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size = 0.2, random_state = 42)



## 모델 은닉층이 0~5개일 때 성능 변화 관찰

results = {} # 은닉층 개수에 따른 모델 정확도를 저장할 딕셔너리

for i in range (6):

  print(f"\n--- 현재 은닉층 개수: {i}개 ---")
  model = keras.Sequential()
  model.add(keras.layers.Flatten(input_shape=(28, 28)))

  for j in range (i): # i개 은닉층 추가
    model.add(keras.layers.Dense(100, activation = 'relu'))

  model.add(keras.layers.Dense(10, activation = 'softmax'))
  model.compile(optimizer = 'RMSprop', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
  model.fit(train_scaled, train_target, epochs = 10, verbose = 0) # 훈련 과정을 생략함으로써 결과만 간단히 출력

  # 검증 세트 점수 저장
  loss, accuracy = model.evaluate(val_scaled, val_target, verbose=0)
  results[f"Layers_{i}"] = accuracy
  print(f"은닉층 {i}개 모델 검증 정확도: {accuracy:.4f}")



## 결과 요약
print("\n" + "="*30)
print("은닉층 개수별 정확도 요약")

for layer, acc in results.items():
  print(f"{layer}: {acc:.4f}")

count_list = [0, 1, 2, 3, 4, 5]

import matplotlib.pyplot as plt
plt.figure(figsize = (10, 6))
plt.plot(count_list, results.values())
plt.title('Accuracy vs. Number of Hidden Layers (Optimizer: RMSprop)')
plt.show()